## Local chemistry and quantum execution


# Build molecular evolution in FlagQuantum

The physical input is the original water geometry and active-space choice. Native
chemistry code computes the Hamiltonian; it is not a pasted circuit. The complete
551-term, 12-qubit operator is retained. The quantum function below requests one
first-order Suzuki-Trotter step at time one. The compiler constructs the Pauli
rotations within width, operation and memory budgets.

`run_workflow` keeps the expensive chemistry and independent reference in
[workflow.py](workflow.py). After it returns, the visible function synthesizes
the same freshly computed operator, executes it and reads named state outcomes.
That makes the high-level quantum call editable without copying integral
computation or validation internals into the Notebook.

Matching the independent product-formula Circuit verifies its implementation,
not agreement with the exact exponential of a noncommuting Hamiltonian. No
hardware execution or quantum advantage follows. Try another finite evolution
time; preserve all terms and compare approximation error separately from
compilation correctness.


In [ ]:
from IPython.display import Image, display

from examples.synthesis.tutorials.basics.exponentiation.example_exponentiation.overview.main import report, save_figure
from examples.synthesis.tutorials.basics.exponentiation.example_exponentiation.overview.model import GEOMETRY, TERM_COUNT, WIDTH
from examples.synthesis.tutorials.basics.exponentiation.example_exponentiation.overview.workflow import run_workflow

print("Geometry in angstroms", GEOMETRY)


## Rebuild the original active-space Hamiltonian


In [ ]:
result = run_workflow()
assert len(result.model.terms) == TERM_COUNT
assert len(result.program.registers["state"]) == WIDTH
print("Fresh restricted Hartree–Fock energy:", result.model.rhf_energy)

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QArray, QBit, allocate, qfunc, suzuki_trotter

hamiltonian = result.model.hamiltonian

@qfunc
def evolve(time: float, state: Output[QArray[QBit]]) -> None:
    allocate(12, state)
    suzuki_trotter(hamiltonian, time, 1, 1, state)

visible_program = synth.synthesize(
    evolve, classical_inputs={"time": 1.0}, max_qubits=12, max_workspace=0,
    max_operations=50_000, memory_limit_bytes=64*1024**2,
)
visible_result = fq.run(visible_program)
print("Named outcomes:", dict(visible_program.probabilities(visible_result, registers=("state",))))
print("Synthesized resources:", visible_program.report["resources"])


## Synthesize the first-order evolution and measure resources


In [ ]:
report(result)
display(Image(filename=str(save_figure(result))))
